<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation Audit

Check whether validation avoids client overlap and whether reported ranking metrics are evaluated on held-out clients.

Run the cells from top to bottom. The source dataset is the anonymized public starter sample; do not add private client data or credentials to this notebook.


## 1. Load the public sample and prepare a conservative target

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)
if "is_declining_label" in df.columns:
    y = pd.to_numeric(df["is_declining_label"], errors="coerce").fillna(0).astype(int)
elif "trend_direction" in df.columns:
    y = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
else:
    raise KeyError("No recognized target field found.")
group_col = "client_id" if "client_id" in df.columns else ("client_hash_id" if "client_hash_id" in df.columns else None)
if group_col is None:
    raise KeyError("A client/group identifier is required for grouped validation.")
forbidden = {"is_declining_label", "trend_direction", "trend_pct", "client_id", "client_hash_id", "content_id", "content_hash_id"}
features = [c for c in df.select_dtypes(include="number").columns if c not in forbidden]
X = df[features].replace([np.inf, -np.inf], np.nan)
print("Rows:", len(df), "| numeric features:", len(features), "| target rate:", round(float(y.mean()), 4))

Rows: 30000 | numeric features: 29 | target rate: 0.5421


## 2. Split by client and prove there is no overlap

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df[group_col]))
train_groups = set(df.iloc[train_idx][group_col].dropna().astype(str))
test_groups = set(df.iloc[test_idx][group_col].dropna().astype(str))
overlap = train_groups & test_groups
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", len(train_groups), "| Test clients:", len(test_groups))
print("Overlapping clients:", len(overlap))
assert len(overlap) == 0, "Validation leakage: client groups overlap."

Train rows: 23837 | Test rows: 6163
Train clients: 25 | Test clients: 7
Overlapping clients: 0


## 3. Fit a simple baseline model and report held-out metrics

In [3]:
if y.iloc[train_idx].nunique() < 2 or y.iloc[test_idx].nunique() < 2:
    print("The split does not contain both classes in train and test; ROC-AUC is not defined for this sample.")
else:
    model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    print("Held-out ROC-AUC:", round(float(roc_auc_score(y.iloc[test_idx], scores)), 4))
    print("Held-out average precision:", round(float(average_precision_score(y.iloc[test_idx], scores)), 4))

def precision_at_k(labels, scores, k=50):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    k = min(int(k), len(labels))
    return float(labels[np.argsort(scores)[::-1][:k]].mean()) if k else float("nan")
if "scores" in locals():
    for k in [10, 20, 50, 100]:
        print(f"Precision@{k}:", round(precision_at_k(y.iloc[test_idx], scores, k), 4))

Held-out ROC-AUC: 1.0
Held-out average precision: 1.0
Precision@10: 1.0
Precision@20: 1.0
Precision@50: 1.0
Precision@100: 1.0


/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. Validation conclusion

Use grouped holdout metrics for the primary result. Compare against a simple baseline on exactly the same test rows, report the target definition and split seed, and do not treat the decline proxy as proof of causality or as a guarantee of future Google rankings.